# Part 1. 객관식

1. 2

2. 3

3. 3

4. 1

5. 2

6. 3

7. 2

8. 2

9. 1

10. 3

# Part 2. 주관식

1. AI를 통해 업무를 수행할 때 사용되는 단위입니다. 자판기에서 원하는 것을 얻을 때 동전을 사용해 OUTPUT을 얻듯이, 토큰이 필요합니다. 이때 한글과 영어는 토큰 비용이 다르게 처리됩니다.

2. 답변길이가 일정 토큰 길이 이상 출력되면 응답을 제한하는 조치가 필요합니다. 이는 입력 토큰보다 출력 토큰의 단가가 4배이기 때문에 출력 토큰의 길이를 제한하는 것이 효과적이다.

3. `temperature`을 0으로, `max_tokens`를 10으로 수정해야 합니다. 문의 분류 기능이기 때문에 매번 다른 답변이 나오면 안되므로 `temperature`는 낮게 설정해야 합니다. 또한 답변 길이를 짧게 통제해야 하기때문에 `max_tokens`를 30으로 제한해야 합니다.

4. 후보 B가 적합합니다. 후보 A에는 페르소나 설정(회의록 요약 비서)과 답변의 특정 형식을 지정하는 내용이 부족합니다.

5. 역할, 말투, 사용할 어휘/금지할 어휘가 반드시 들어가야 합니다. 역할은 해당 AI가 어떤 역할을 수행해야하 하는지 지정하여 성격을 부여합니다. 말투는 해당 역할을 수행하면서 사용자에게 해당 캐릭터의 설득력을 더 높여주는 역할을 합니다. 또한 해당 페르소나에서 적절한 어휘를 사용해야 하므로 사용할 어휘/금지할 어휘를 지정해줄 필요가 있습니다.

In [2]:
# 6. 첫 API 호출 구현

# OpenAI 클라이언트 생성
from openai import OpenAI

client = OpenAI()  

# 응답 생성
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {"role": "user", "content":"RAG가 무엇인가요?"}
    ]
)

# 응답에서 답변 텍스트만 출력
print(response.choices[0].message.content)



RAG는 "Red, Amber, Green"의 약자로, 주로 프로젝트 관리나 성과 관리에서 사용되는 색상 기반의 평가 시스템을 의미합니다. 각 색상은 특정 상태나 상황을 나타냅니다:

- **Red (빨간색)**: 심각한 문제나 위험이 있으며, 즉각적인 조치가 필요함을 의미합니다.
- **Amber (주황색)**: 위험이 있으며, 주의가 필요하지만 즉각적인 조치는 필요하지 않은 상태를 나타냅니다.
- **Green (초록색)**: 모든 것이 정상적으로 진행되고 있으며, 문제가 없음을 나타냅니다.

이 시스템은 주간 또는 월간 보고서에서 프로젝트의 진행 상황을 간단하고 명확하게 전달하는 데 유용합니다. RAG 평가는 시각적으로 쉽게 이해할 수 있어 팀원들이나 경영진과의 커뮤니케이션을 효율적으로 도와줍니다.


In [8]:
# 7. 대화 맥락 유지

def ask_llm(messages, model="gpt-4o-mini", temperature=0.3, max_tokens=500):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content
    

messages = [
    {"role": "user", "content": "제 이름은 이서현입니다."}, # 사용자 입력으로 role : user 지정
    {"role": "assistant", "content": "안녕하세요, 이서현님!"},  # llm 의 등장으로 role : assistant 지정
    {"role": "user", "content": "제 이름이 뭐였죠?"} # 사용자 입력으로 role : user 지정
]

print(ask_llm(messages))

당신의 이름은 이서현입니다!


In [ ]:
# 8. 고객 문의 분류기 완성
def classify_inquiry(text: str) -> str:
    messages = [
        {"role": "system", "content": "당신은 고객센터 상담사입니다. 고객의 문의를 '배송', '결제', '반품', '기타' 중 하나로 분류해 주세요."},
        {"role": "user", "content": text}
    ]
    return ask_llm(messages, temperature=0.0, max_tokens=10)

test_cases = [
    "어제 주문했는데 카드 결제가 두 번 되었어요.",
    "택배가 일주일째 안 와요",
    "사이즈가 안 맞아서 반품하고 싶어요.",
    "영업시간이 어떻게 되나요?"
]

for t in test_cases:
    category = classify_inquiry(t)
    print(f"문의: {t} => 분류: {category}")

문의: 어제 주문했는데 카드 결제가 두 번 되었어요. => 분류: 이 문의는 '결제'로 분류
문의: 택배가 일주일째 안 와요 => 분류: 배송
문의: 사이즈가 안 맞아서 반품하고 싶어요. => 분류: 반품으로 분류됩니다.
문의: 영업시간이 어떻게 되나요? => 분류: 기타


In [12]:
# 9. temperature 비교 실험
prompt = "AI 스타트업 이름 3개를 지어주세요"

for temp in [0, 1.2]:
    answer = ask_llm([{"role": "user", "content": prompt}], temperature=temp)

    print(f"temperature={temp}\n -> {answer}\n")


temperature=0
 -> 물론입니다! 다음은 AI 스타트업을 위한 이름 세 가지입니다:

1. **인사이트AI (InsightAI)** - 데이터 분석과 인사이트 제공에 중점을 둔 스타트업에 적합한 이름입니다.
  
2. **넥스트젠AI (NextGenAI)** - 차세대 기술과 혁신을 강조하는 이름으로, 미래 지향적인 AI 솔루션을 제공하는 스타트업에 어울립니다.

3. **스마트코어 (SmartCore)** - AI의 핵심 기술을 바탕으로 다양한 산업에 적용할 수 있는 솔루션을 개발하는 스타트업에 적합한 이름입니다.

이 이름들이 도움이 되길 바랍니다!

temperature=1.2
 -> 물론입니다! 다음은 AI 스타트업에 어울리는 3가지 이름입니다.

1. **AIvance** (AI + Advance의 조합) - 인공지능을 통해 혁신과 발전을 선도하는 이미지를 추구하는 이름입니다.

2. **NeuraNest** - "Neura"는 신경망을 의미하고, "Nest"는 아이디어와 창의력이 자라는 공간을 상징합니다. AI 관련 연구와 개발에 어울리는 이름입니다.

3. **Cognify** - "Cognitive"와 "Amplify"의 결합으로, 인식하는 능력을 배가시키고 발전시키는 서비스를 제공하는 스타트업에 적합합니다.

각 이름은 독창성과 혁신을 강조하며, 스타트업의 목표와 비전을 반영할 수 있습니다. 도움이 되길 바랍니다!



temperature=0 는 말투가 단조롭고 정직한 이름은 만들어줬다면 temperature=1.2는 좀 더 창의적이고 다양한 이름을 제시해주었습니다.

In [13]:
# 10. 호출 비용 계산

PRICE_INPUT = 0.15
PRICE_OUTPUT = 0.60
USD_TO_KRW = 1400

def cost_korean_won(usage):
    input_cost = usage.prompt_tokens / 1_000_000 * PRICE_INPUT
    output_cost = usage.completion_tokens / 1_000_000 * PRICE_OUTPUT
    return (input_cost + output_cost) * USD_TO_KRW

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Python의 장점을 5가지 설명해주세요."}],
)


u = response.usage
print(f"입력 {u.prompt_tokens} + 출력 {u.completion_tokens} = "
        f"총 {u.total_tokens} 토큰, 비용 약 {cost_korean_won(u):.4f}원")

입력 17 + 출력 413 = 총 430 토큰, 비용 약 0.3505원
